-> Duplicate the categorical data
-> Ignore the categorical data
-> For the numerical find the range and make sure that the synthetic value generated is in that range.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.cluster import MiniBatchKMeans
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import InstanceHardnessThreshold
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.ensemble import IsolationForest
from imblearn.ensemble import BalancedBaggingClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score, average_precision_score
)

In [2]:
df = pd.read_csv(r"C:\Users\ankit\OneDrive\Desktop\Research_Papers\Online_Payments_Dataset.zip")
df.info()                     
df.describe()                 
df.nunique()                   
df.isnull().sum()              

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [3]:
df['orig_is_merchant'] = df['nameOrig'].str.startswith('M').astype(int)
df['dest_is_merchant'] = df['nameDest'].str.startswith('M').astype(int)

df = pd.get_dummies(df, columns=['type'], drop_first=True)

bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

df = df.drop(columns=['nameOrig', 'nameDest'])

print(df.shape)
print(df.select_dtypes(include='object').columns)
df.dtypes

(6362620, 14)
Index([], dtype='object')


step                  int64
amount              float64
oldbalanceOrg       float64
newbalanceOrig      float64
oldbalanceDest      float64
newbalanceDest      float64
isFraud               int64
isFlaggedFraud        int64
orig_is_merchant      int64
dest_is_merchant      int64
type_CASH_OUT         int64
type_DEBIT            int64
type_PAYMENT          int64
type_TRANSFER         int64
dtype: object

In [4]:
X = df.drop(columns=['isFraud'])
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train set fraud cases:", y_train.sum(), "/", len(y_train))
print("Test set fraud cases:", y_test.sum(), "/", len(y_test), "\n")

Train set fraud cases: 6570 / 5090096
Test set fraud cases: 1643 / 1272524 



In [5]:
categorical_like_cols = ['orig_is_merchant', 'dest_is_merchant'] \
    + [c for c in X_train.columns if c.startswith('type_')]

if 'isFlaggedFraud' in X_train.columns:
    categorical_like_cols.append('isFlaggedFraud')

numeric_cols = [c for c in X_train.columns if c not in categorical_like_cols]
print("Numeric columns used for SMOTE:", numeric_cols)
print("Categorical columns excluded:", categorical_like_cols)

Numeric columns used for SMOTE: ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
Categorical columns excluded: ['orig_is_merchant', 'dest_is_merchant', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER', 'isFlaggedFraud']


In [6]:
numeric_ranges = {
    col: (X_train[col].min(), X_train[col].max()) for col in numeric_cols
}
print("\nNumeric ranges:")
for k, v in numeric_ranges.items():
    print(f"  {k}: {v}")


Numeric ranges:
  step: (np.int64(1), np.int64(743))
  amount: (np.float64(0.0), np.float64(92445516.64))
  oldbalanceOrg: (np.float64(0.0), np.float64(59585040.37))
  newbalanceOrig: (np.float64(0.0), np.float64(49585040.37))
  oldbalanceDest: (np.float64(0.0), np.float64(356015889.35))
  newbalanceDest: (np.float64(0.0), np.float64(356179278.92))


In [7]:
X_train_numeric = X_train[numeric_cols].reset_index(drop=True)
y_train_reset = y_train.reset_index(drop=True)

sm = SMOTE(k_neighbors=3, random_state=42)
X_train_num_res, y_train_res = sm.fit_resample(X_train_numeric, y_train_reset)

print("\nClass balance after SMOTE:", pd.Series(y_train_res).value_counts().to_dict())



Class balance after SMOTE: {0: 5083526, 1: 5083526}


In [8]:
n_original = len(X_train_numeric)
n_total = len(X_train_num_res)

is_synthetic = np.zeros(n_total, dtype=bool)
is_synthetic[n_original:] = True

synthetic_df = X_train_num_res.iloc[n_original:].copy()

def in_range(row):
    return all(
        numeric_ranges[col][0] <= row[col] <= numeric_ranges[col][1]
        for col in numeric_cols
    )

valid_mask = synthetic_df.apply(in_range, axis=1)

print(f"\nTotal synthetic samples generated: {len(synthetic_df)}")
print(f"Valid (in-range) synthetic samples: {valid_mask.sum()}")
print(f"Invalid (out-of-range) synthetic samples dropped: {(~valid_mask).sum()}")


Total synthetic samples generated: 5076956
Valid (in-range) synthetic samples: 5076956
Invalid (out-of-range) synthetic samples dropped: 0


In [9]:
valid_synthetic_df = synthetic_df[valid_mask]
valid_synthetic_y = y_train_res.iloc[n_original:][valid_mask]

X_train_numeric_final = pd.concat(
    [X_train_numeric, valid_synthetic_df], axis=0
).reset_index(drop=True)

y_train_final = pd.concat(
    [y_train_reset, valid_synthetic_y], axis=0
).reset_index(drop=True)

print("\nFinal class balance after filtering:",
      y_train_final.value_counts().to_dict())


Final class balance after filtering: {0: 5083526, 1: 5083526}


In [11]:
X_test_numeric = X_test[numeric_cols]

clf_brf = BalancedRandomForestClassifier(
    n_estimators=150,
    max_depth=2,          
    sampling_strategy='all', 
    replacement=True,  
    random_state=42,
    n_jobs=-1            
)
clf_brf.fit(X_train_numeric_final, y_train_final)

y_pred = clf_brf.predict(X_test_numeric)

print("Classification Report:")
print(classification_report(y_test, y_pred, digits=3))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

MemoryError: Unable to allocate 77.6 MiB for an array with shape (10167052,) and data type int64

In [ ]:
for depth in [2, 4, 6, None]:
    clf = BalancedRandomForestClassifier(
        n_estimators=150,
        max_depth=depth,
        sampling_strategy='all',
        replacement=True,
        random_state=42,
        n_jobs=-1
    )
    clf.fit(X_train_numeric_final, y_train_final)
    y_pred = clf.predict(X_test_numeric)
    print(f"\n--- max_depth={depth} ---")
    print(classification_report(y_test, y_pred, digits=3))